## Step 1: Install Required Libraries

In [ ]:
!pip install playwright
!playwright install chromium
!playwright install-deps

## Step 2: Import Libraries

In [ ]:
from playwright.sync_api import sync_playwright
import os
import re
import time
import zipfile
from datetime import datetime
import base64
from IPython.display import HTML, display

print("✅ Libraries imported successfully!")

## Step 3: Configure Website & Login Details

⚡ **EDIT THIS CONFIG - ONLY PLACE TO CHANGE!**

In [ ]:
CONFIG = {
    # Website login URL
    "login_url": "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/login",
    
    # Login credentials
    "username": "ashish.admin",
    "password": "Staging123$",
    
    # Login form selectors (CSS selectors for username, password, submit button)
    # Leave empty "" to auto-detect common patterns
    "username_selector": "input[type='text'], input[name='username'], input[id='username']",
    "password_selector": "input[type='password'], input[name='password'], input[id='password']",
    "submit_selector": "button[type='submit'], input[type='submit'], button:has-text('Login')",
    
    # Pages to screenshot (list of URLs to visit after login)
    # Leave empty [] to only screenshot the page after login
    "pages_to_screenshot": [
        # Add more URLs here, e.g.:
        # "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/dashboard",
        # "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/users",
    ],
    
    # Wait time between actions (seconds)
    "wait_after_login": 3,      # Wait after login completes
    "wait_before_screenshot": 2, # Wait before taking each screenshot
    "wait_between_pages": 2,     # Wait between navigating to pages
    
    # Screenshot settings
    "full_page": True,  # Capture full scrollable page (True) or just viewport (False)
    "output_folder": "screenshots",
    "zip_filename": "website_screenshots.zip",
    
    # Browser settings
    "headless": False,  # Set to True to hide browser window
}

print("📋 Configuration loaded:")
print(f"  🌐 Login URL: {CONFIG['login_url']}")
print(f"  👤 Username: {CONFIG['username']}")
print(f"  🔒 Password: {'*' * len(CONFIG['password'])}")
print(f"  📸 Pages to screenshot: {len(CONFIG['pages_to_screenshot']) + 1} (login page + additional)")
print(f"  📁 Output: {CONFIG['zip_filename']}")
print("\n✅ Ready to start!")

## Step 4: Run Screenshot Automation

This will:
1. Open browser and navigate to login page
2. Fill credentials and login
3. Take screenshots of all specified pages
4. Create ZIP file with all screenshots
5. Provide download link

In [ ]:
def sanitize_filename(url):
    """Convert URL to safe filename."""
    # Remove protocol and special characters
    name = re.sub(r'https?://', '', url)
    name = re.sub(r'[^a-zA-Z0-9_-]', '_', name)
    # Limit length
    if len(name) > 100:
        name = name[:100]
    return name

def take_screenshots(config):
    """Main function to automate login and take screenshots."""
    
    # Create output folder
    os.makedirs(config['output_folder'], exist_ok=True)
    
    screenshots_taken = []
    
    print("\n" + "="*70)
    print("🚀 Starting Screenshot Automation...")
    print("="*70 + "\n")
    
    with sync_playwright() as p:
        # Launch browser
        print("🌐 Launching browser...")
        browser = p.chromium.launch(headless=config['headless'])
        context = browser.new_context(
            viewport={'width': 1920, 'height': 1080}
        )
        page = context.new_page()
        
        try:
            # Navigate to login page
            print(f"📍 Navigating to: {config['login_url']}")
            page.goto(config['login_url'], wait_until='networkidle')
            time.sleep(2)
            
            # Fill login form
            print("\n🔐 Logging in...")
            
            # Find and fill username
            username_field = page.locator(config['username_selector']).first
            username_field.fill(config['username'])
            print(f"  ✓ Username entered: {config['username']}")
            
            # Find and fill password
            password_field = page.locator(config['password_selector']).first
            password_field.fill(config['password'])
            print(f"  ✓ Password entered: {'*' * len(config['password'])}")
            
            # Click submit button
            submit_button = page.locator(config['submit_selector']).first
            submit_button.click()
            print("  ✓ Login button clicked")
            
            # Wait for navigation after login
            print(f"\n⏳ Waiting {config['wait_after_login']} seconds for login to complete...")
            time.sleep(config['wait_after_login'])
            
            current_url = page.url
            print(f"✅ Login successful! Current URL: {current_url}")
            
            # Take screenshot of post-login page
            print(f"\n📸 Taking screenshot of post-login page...")
            time.sleep(config['wait_before_screenshot'])
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{sanitize_filename(current_url)}_{timestamp}.png"
            filepath = os.path.join(config['output_folder'], filename)
            
            page.screenshot(path=filepath, full_page=config['full_page'])
            screenshots_taken.append(filename)
            print(f"  ✓ Saved: {filename}")
            
            # Screenshot additional pages
            if config['pages_to_screenshot']:
                print(f"\n📸 Taking screenshots of {len(config['pages_to_screenshot'])} additional pages...\n")
                
                for idx, url in enumerate(config['pages_to_screenshot'], 1):
                    print(f"  [{idx}/{len(config['pages_to_screenshot'])}] Navigating to: {url}")
                    page.goto(url, wait_until='networkidle')
                    time.sleep(config['wait_before_screenshot'])
                    
                    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                    filename = f"{sanitize_filename(url)}_{timestamp}.png"
                    filepath = os.path.join(config['output_folder'], filename)
                    
                    page.screenshot(path=filepath, full_page=config['full_page'])
                    screenshots_taken.append(filename)
                    print(f"  ✓ Saved: {filename}")
                    
                    if idx < len(config['pages_to_screenshot']):
                        time.sleep(config['wait_between_pages'])
            
        except Exception as e:
            print(f"\n❌ Error occurred: {e}")
            print("\n💡 Troubleshooting tips:")
            print("  1. Check if login selectors are correct")
            print("  2. Verify username/password are correct")
            print("  3. Website might require CAPTCHA or 2FA")
            print("  4. Try increasing wait times in CONFIG")
            raise
        
        finally:
            # Close browser
            browser.close()
            print("\n🔒 Browser closed")
    
    return screenshots_taken

def create_zip(config, screenshots):
    """Create ZIP file with all screenshots."""
    print("\n" + "="*70)
    print("📦 Creating ZIP file...")
    print("="*70)
    
    zip_path = config['zip_filename']
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for screenshot in screenshots:
            filepath = os.path.join(config['output_folder'], screenshot)
            zipf.write(filepath, screenshot)
            print(f"  ✓ Added: {screenshot}")
    
    file_size = os.path.getsize(zip_path)
    file_size_mb = file_size / (1024 * 1024)
    
    print(f"\n✅ ZIP created successfully!")
    print(f"  📄 Filename: {zip_path}")
    print(f"  📊 Size: {file_size_mb:.2f} MB ({file_size:,} bytes)")
    print(f"  📸 Screenshots: {len(screenshots)}")
    
    return zip_path, file_size_mb

def create_download_link(zip_path, file_size_mb):
    """Create download link for the ZIP file."""
    print("\n" + "="*70)
    print("🔗 Creating download link...")
    print("="*70 + "\n")
    
    # Check if running in Google Colab
    try:
        from google.colab import files
        print("📍 Google Colab detected - using files.download()\n")
        files.download(zip_path)
        print("✅ Download started! Check your browser downloads.")
        return
    except ImportError:
        pass
    
    # For VS Code or local Jupyter - create base64 link
    print("📍 Local environment detected - creating download link\n")
    
    if file_size_mb > 50:
        print("⚠️  WARNING: File is larger than 50MB")
        print("   Creating download link may take a while...\n")
    
    # Read and encode file
    with open(zip_path, 'rb') as f:
        zip_data = f.read()
    
    b64_data = base64.b64encode(zip_data).decode()
    
    # Create download link
    download_link = f'<a href="data:application/zip;base64,{b64_data}" download="{zip_path}">Click here to download {zip_path}</a>'
    
    html = f"""
    <div style="padding: 20px; border: 2px solid #4CAF50; border-radius: 10px; background-color: #f9f9f9;">
        <h2 style="color: #4CAF50; margin-top: 0;">✅ Screenshots Ready!</h2>
        <p style="font-size: 16px;"><strong>File:</strong> {zip_path}</p>
        <p style="font-size: 16px;"><strong>Size:</strong> {file_size_mb:.2f} MB</p>
        <div style="margin-top: 20px;">
            {download_link}
        </div>
    </div>
    """
    
    display(HTML(html))
    print("\n✅ Download link created! Click the link above to download.")

# Run the automation
try:
    screenshots = take_screenshots(CONFIG)
    zip_path, file_size_mb = create_zip(CONFIG, screenshots)
    create_download_link(zip_path, file_size_mb)
    
    print("\n" + "="*70)
    print("🎉 ALL DONE!")
    print("="*70)
    print(f"\n📦 {len(screenshots)} screenshot(s) packaged successfully!")
    print(f"📁 Location: {CONFIG['output_folder']}/")
    print(f"📥 ZIP File: {zip_path}")
    print("\n💡 TIP: You can edit CONFIG above to add more pages to screenshot!")
    
except Exception as e:
    print(f"\n❌ Error: {e}")

## 💡 How to Add More Pages to Screenshot

Edit the `CONFIG` in **Step 3** and add URLs to the `pages_to_screenshot` list:

```python
"pages_to_screenshot": [
    "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/dashboard",
    "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/users",
    "https://rms-ui-dev-slp.supremelifeplatform.com/#/admin/settings",
],
```

Then just re-run **Step 4**!

## 🔧 Troubleshooting

### Login not working?
1. Check if selectors in CONFIG are correct
2. Inspect the login page and update selectors
3. Try setting `headless: False` to see what's happening

### Screenshots incomplete?
1. Increase `wait_before_screenshot` in CONFIG
2. Increase `wait_after_login` for slow-loading pages

### File too large?
- ZIP files >50MB may take time to encode for download
- Consider taking fewer screenshots or lower resolution